# MYSTIC MOTION — Marvel-inspired Webcam Magic VFX

Real-time webcam hand tracking with six character modes and procedural visual effects.

**Character controls:** `1` Doctor Strange · `2` Scarlet Witch (Wanda) · `3` Agatha · `4` Loki · `5` Thor · `6` Thanos

**Runtime controls:** `H` show/hide help · `S` show/hide hand skeleton · `Q` quit

**Thanos:** Select `6`, then perform a finger snap (bring thumb and index finger together, then quickly flick them apart). The detected person is segmented and dissolves out. Snap again to return.

The effects are original OpenCV approximations, not extracted movie footage or pixel-identical film assets. The webcam feed is processed locally. Allow camera access when prompted.

Run the cells from top to bottom. The final cell starts the webcam experience.


In [1]:
%pip install -q "mediapipe==0.10.14" "opencv-python>=4.9.0" "numpy>=1.24,<2.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

for folder in ("src", "models", "assets"):
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Project folders ready:", Path.cwd())

Project folders ready: /Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM


In [3]:
%%writefile src/config.py
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class Character:
    key: str
    name: str
    bgr: tuple[int, int, int]
    accent: tuple[int, int, int]

# Six character modes. OpenCV uses BGR colour order.
CHARACTERS = {
    "1": Character("1", "DOCTOR STRANGE", (0, 155, 255), (80, 220, 255)),
    "2": Character("2", "SCARLET WITCH", (0, 0, 235), (70, 35, 255)),
    "3": Character("3", "AGATHA HARKNESS", (220, 35, 235), (245, 145, 255)),
    "4": Character("4", "LOKI", (45, 205, 55), (145, 255, 155)),
    "5": Character("5", "THOR", (255, 205, 65), (255, 250, 225)),
    "6": Character("6", "THANOS", (150, 80, 205), (205, 155, 255)),
}

# MediaPipe Gesture Recognizer model configuration
PROJECT_ROOT = Path(__file__).resolve().parent.parent
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODEL_DIR / "gesture_recognizer.task"
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "gesture_recognizer/gesture_recognizer/float16/1/"
    "gesture_recognizer.task"
)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

Overwriting src/config.py


In [4]:
%%writefile src/hand_tracking.py
from pathlib import Path
from urllib.request import urlretrieve
import time

import cv2
import mediapipe as mp
import numpy as np

from config import MODEL_PATH, MODEL_URL

class HandTracker:
    """Pretrained MediaPipe Gesture Recognizer; no project-specific training."""

    def __init__(self, max_hands: int = 2):
        model_path = Path(MODEL_PATH)
        model_path.parent.mkdir(parents=True, exist_ok=True)
        if not model_path.exists():
            print("Downloading pretrained MediaPipe Gesture Recognizer...")
            urlretrieve(MODEL_URL, model_path)

        BaseOptions = mp.tasks.BaseOptions
        GestureRecognizer = mp.tasks.vision.GestureRecognizer
        GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
        RunningMode = mp.tasks.vision.RunningMode

        options = GestureRecognizerOptions(
            base_options=BaseOptions(model_asset_path=str(model_path)),
            running_mode=RunningMode.VIDEO,
            num_hands=max_hands,
            min_hand_detection_confidence=0.55,
            min_hand_presence_confidence=0.50,
            min_tracking_confidence=0.50,
        )
        self.recognizer = GestureRecognizer.create_from_options(options)
        self.last_timestamp_ms = 0

    def process(self, bgr_frame):
        rgb = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        timestamp_ms = int(time.monotonic() * 1000)
        timestamp_ms = max(timestamp_ms, self.last_timestamp_ms + 1)
        self.last_timestamp_ms = timestamp_ms

        result = self.recognizer.recognize_for_video(mp_image, timestamp_ms)
        hands = []
        for index, landmarks in enumerate(result.hand_landmarks):
            points = np.asarray(
                [[p.x, p.y, p.z] for p in landmarks], dtype=np.float32
            )
            gesture = "None"
            score = 0.0
            if index < len(result.gestures) and result.gestures[index]:
                gesture = result.gestures[index][0].category_name
                score = float(result.gestures[index][0].score)

            handedness = "Unknown"
            if index < len(result.handedness) and result.handedness[index]:
                handedness = result.handedness[index][0].category_name

            hands.append({
                "points": points,
                "gesture": gesture,
                "confidence": score,
                "handedness": handedness,
            })
        return hands

    def close(self):
        self.recognizer.close()


class PersonSegmenter:
    """Segments the person in the webcam frame for the Thanos disappearance effect."""

    def __init__(self):
        selfie = getattr(mp, "solutions", None)
        if selfie is None or not hasattr(selfie, "selfie_segmentation"):
            raise RuntimeError(
                "This MediaPipe build does not expose selfie_segmentation. "
                "Install mediapipe==0.10.14 and restart the kernel."
            )
        self.segmenter = selfie.selfie_segmentation.SelfieSegmentation(
            model_selection=1
        )

    def mask(self, bgr_frame):
        rgb = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
        result = self.segmenter.process(rgb)
        if result.segmentation_mask is None:
            return None
        return result.segmentation_mask

    def close(self):
        self.segmenter.close()


Overwriting src/hand_tracking.py


In [5]:
%%writefile src/vfx.py
import math
import random
from collections import deque

import cv2
import numpy as np

from config import Character


# ---------------------------------------------------------------------
# Shared drawing primitives. Each character uses its own visual language;
# there is deliberately no generic finger-to-finger lightning/line effect.
# ---------------------------------------------------------------------

def palm_center(points):
    return np.mean(points[[0, 5, 9, 13, 17], :2], axis=0)


def to_px(point, width, height):
    return (
        int(np.clip(point[0] * width, 0, width - 1)),
        int(np.clip(point[1] * height, 0, height - 1)),
    )


def hand_is_open(points):
    wrist = points[0, :2]
    checks = ((8, 6), (12, 10), (16, 14), (20, 18))
    extended = sum(
        np.linalg.norm(points[tip, :2] - wrist)
        > np.linalg.norm(points[pip, :2] - wrist) * 1.12
        for tip, pip in checks
    )
    return extended >= 3


def is_pinching(points, threshold=0.060):
    return float(np.linalg.norm(points[4, :2] - points[8, :2])) < threshold


def poly(layer, points, color, thickness=2, closed=False):
    if len(points) >= 2:
        pts = np.asarray(points, dtype=np.int32)
        cv2.polylines(layer, [pts], closed, color, thickness, cv2.LINE_AA)


def glow(layer, center, radius, color, core=(245, 248, 255), strength=1.0):
    x, y = center
    cv2.circle(layer, (x, y), max(2, int(radius * 2.8)), color, -1, cv2.LINE_AA)
    cv2.circle(layer, (x, y), max(1, int(radius)), core, -1, cv2.LINE_AA)


def bloom_composite(frame, overlay, blur=13, bloom_strength=0.8, detail_strength=0.95):
    soft = cv2.GaussianBlur(overlay, (0, 0), blur)
    frame[:] = cv2.addWeighted(frame, 1.0, soft, bloom_strength, 0)
    frame[:] = cv2.addWeighted(frame, 1.0, overlay, detail_strength, 0)


def mandala(layer, center, radius, color, accent, t):
    """Layered, symmetrical sorcerer mandala: clean geometry, ticks and runes."""
    cx, cy = center
    radius = max(34, int(radius))
    # Fine concentric rings with intentional gaps.
    for i, scale in enumerate((1.00, 0.91, 0.78, 0.64, 0.48, 0.30)):
        r = int(radius * scale)
        start = int((t * (18 if i % 2 else -14) + i * 17) % 360)
        cv2.ellipse(layer, center, (r, r), 0, start, start + 350,
                    color if i % 2 == 0 else accent, 2 if i < 3 else 1, cv2.LINE_AA)

    # Radial divisions and small rune-like rectangular marks.
    for i in range(48):
        a = math.tau * i / 48 + t * 0.025
        r1 = radius * (0.82 if i % 4 else 0.75)
        r2 = radius * (0.91 if i % 4 else 1.04)
        p1 = (int(cx + math.cos(a) * r1), int(cy + math.sin(a) * r1))
        p2 = (int(cx + math.cos(a) * r2), int(cy + math.sin(a) * r2))
        cv2.line(layer, p1, p2, accent if i % 4 == 0 else color,
                 2 if i % 4 == 0 else 1, cv2.LINE_AA)

    # Nested rotated squares / octagonal geometry.
    for sides, scale, rotation in ((4, 0.69, 45), (4, 0.52, 0), (8, 0.39, 22.5)):
        pts = []
        for i in range(sides):
            a = math.tau * i / sides + math.radians(rotation) + t * 0.035
            r = radius * scale
            pts.append((int(cx + math.cos(a) * r), int(cy + math.sin(a) * r)))
        poly(layer, pts, color, 2, True)

    # Short rune strokes around the outer band; kept separate, not connected.
    for i in range(24):
        a = math.tau * i / 24 + t * 0.018
        r = radius * 0.96
        x, y = int(cx + math.cos(a) * r), int(cy + math.sin(a) * r)
        tx, ty = -math.sin(a), math.cos(a)
        length = 5 if i % 3 else 10
        cv2.line(layer, (int(x - tx * length), int(y - ty * length)),
                 (int(x + tx * length), int(y + ty * length)),
                 accent, 1, cv2.LINE_AA)

    # Bright central seal.
    cv2.circle(layer, center, max(8, int(radius * 0.16)), color, 2, cv2.LINE_AA)
    cv2.circle(layer, center, max(3, int(radius * 0.07)), accent, 2, cv2.LINE_AA)

def electric_bolt(layer, start, end, color, seed, amplitude=16, branches=2, thickness=2):
    """Jagged branching lightning with a bright white-hot core."""
    rng = random.Random(seed)
    a = np.asarray(start, dtype=float)
    b = np.asarray(end, dtype=float)
    delta = b - a
    length = np.linalg.norm(delta) + 1e-6
    perp = np.array([-delta[1], delta[0]]) / length
    count = max(5, int(length / 24))
    pts = [tuple(a.astype(int))]
    for i in range(1, count):
        f = i / count
        offset = rng.uniform(-amplitude, amplitude)
        q = a + delta * f + perp * offset
        pts.append(tuple(q.astype(int)))
    pts.append(tuple(b.astype(int)))
    poly(layer, pts, color, thickness)
    poly(layer, pts, (245, 250, 255), max(1, thickness // 2))
    if branches:
        for j in range(branches):
            idx = rng.randint(1, max(2, len(pts) - 2))
            origin = pts[idx]
            branch_end = (
                int(origin[0] + rng.uniform(-35, 35)),
                int(origin[1] + rng.uniform(-42, 42)),
            )
            electric_bolt(layer, origin, branch_end, color, seed + j + 91,
                          amplitude * 0.45, 0, max(1, thickness - 1))


def energy_wisp(layer, start, end, color, t, seed=0, width=3):
    """Soft flowing red/purple energy filament, not a straight connector."""
    rng = random.Random(seed)
    a = np.asarray(start, dtype=float)
    b = np.asarray(end, dtype=float)
    delta = b - a
    length = np.linalg.norm(delta) + 1e-6
    perp = np.array([-delta[1], delta[0]]) / length
    pts = [tuple(a.astype(int))]
    count = max(8, int(length / 18))
    phase = rng.random() * math.tau
    for i in range(1, count):
        f = i / count
        envelope = math.sin(math.pi * f)
        offset = math.sin(t * 5.0 + f * 15.0 + phase) * 13 * envelope
        offset += math.sin(t * 8.0 + f * 29.0 + phase) * 4 * envelope
        q = a + delta * f + perp * offset
        pts.append(tuple(q.astype(int)))
    pts.append(tuple(b.astype(int)))
    poly(layer, pts, color, width + 3)
    poly(layer, pts, (255, 235, 245), max(1, width - 1))


class MagicRenderer:
    """Six distinct, webcam-driven magic styles based on the supplied references.

    These are procedural real-time VFX approximations, not original movie assets.
    """

    def __init__(self):
        self.history = {}

    def _remember(self, key, point, size=12):
        trail = self.history.setdefault(key, deque(maxlen=size))
        trail.append(point)
        return list(trail)

    def _render_strange(self, overlay, palm, elapsed, power):
        radius = int(78 + 20 * power)
        mandala(overlay, palm, radius, (0, 155, 255), (80, 220, 255), elapsed)
        # Sparse sparks on the circumference only.
        for i in range(8):
            a = elapsed * 0.22 + i * math.tau / 8
            q = (int(palm[0] + math.cos(a) * radius * 1.08),
                 int(palm[1] + math.sin(a) * radius * 1.08))
            cv2.circle(overlay, q, 2, (90, 225, 255), -1, cv2.LINE_AA)
        glow(overlay, palm, 7, (0, 165, 255), (255, 245, 220))

    def _render_wanda(self, overlay, palm, tips, elapsed, open_hand):
        # Scarlet Witch: a turbulent crimson core with rotating chaos sigils.
        pulse = 1.0 + 0.22 * math.sin(elapsed * 8.0)
        glow(overlay, palm, int(24 * pulse), (0, 0, 245), (255, 225, 245))
        cv2.circle(overlay, palm, 34, (0, 0, 205), 2, cv2.LINE_AA)
        cv2.ellipse(overlay, palm, (49, 31), elapsed * 34, 18, 315,
                    (0, 0, 245), 2, cv2.LINE_AA)
        cv2.ellipse(overlay, palm, (57, 40), -elapsed * 27, 205, 500,
                    (35, 25, 220), 2, cv2.LINE_AA)

        # Red tendrils flare outward, then curl back toward the palm.
        active = tips[1:] if open_hand else [tips[1]]
        for i, tip in enumerate(active):
            energy_wisp(overlay, palm, tip, (0, 0, 245),
                        elapsed + i * 0.12, seed=40 + i, width=4)
        for i in range(7):
            angle = elapsed * 0.9 + i * math.tau / 7
            end = (int(palm[0] + math.cos(angle) * 62),
                   int(palm[1] + math.sin(angle) * 47))
            energy_wisp(overlay, palm, end, (0, 0, 205),
                        elapsed + i * 0.08, seed=120 + i, width=2)
        for i in range(12):
            angle = elapsed * 0.45 + i * math.tau / 12
            x = int(palm[0] + math.cos(angle) * 68)
            y = int(palm[1] + math.sin(angle) * 53)
            cv2.circle(overlay, (x, y), 2, (65, 55, 255), -1, cv2.LINE_AA)

    def _render_agatha(self, overlay, palm, tips, elapsed, open_hand):
        # Agatha: deep violet witchfire, layered coils and a dark-purple corona.
        pulse = 1.0 + 0.15 * math.sin(elapsed * 5.5)
        glow(overlay, palm, int(20 * pulse), (175, 25, 205), (255, 220, 255))
        cv2.ellipse(overlay, palm, (38, 25), elapsed * 25, 0, 330,
                    (205, 45, 235), 3, cv2.LINE_AA)
        cv2.ellipse(overlay, palm, (52, 37), -elapsed * 19, 30, 350,
                    (145, 25, 195), 2, cv2.LINE_AA)
        for i, tip in enumerate(tips[1:]):
            if open_hand or i == 0:
                energy_wisp(overlay, palm, tip, (190, 35, 225),
                            elapsed + i * 0.18, seed=220 + i, width=3)
        # Separate floating sparks create a smoky, spell-casting halo.
        for i in range(9):
            angle = elapsed * 0.55 + i * math.tau / 9
            radius_x = 58 + 7 * math.sin(elapsed * 2 + i)
            radius_y = 43 + 5 * math.cos(elapsed * 2.3 + i)
            q = (int(palm[0] + math.cos(angle) * radius_x),
                 int(palm[1] + math.sin(angle) * radius_y))
            cv2.circle(overlay, q, 2 + (i % 2), (225, 95, 255), -1, cv2.LINE_AA)
        glow(overlay, palm, 5, (205, 45, 245), (255, 245, 255))

    def _render_loki(self, overlay, palm, tips, elapsed, open_hand):
        # Loki: elegant emerald illusion ribbons, not generic lightning bolts.
        glow(overlay, palm, 16, (35, 175, 45), (220, 255, 220))
        cv2.ellipse(overlay, palm, (43, 28), elapsed * 32, 25, 320,
                    (45, 210, 70), 2, cv2.LINE_AA)
        cv2.ellipse(overlay, palm, (58, 39), -elapsed * 23, 70, 355,
                    (25, 145, 45), 2, cv2.LINE_AA)
        selected = tips[1:] if open_hand else [tips[1]]
        for i, tip in enumerate(selected):
            energy_wisp(overlay, palm, tip, (45, 220, 70),
                        elapsed + i * 0.16, seed=320 + i, width=3)
        # Small rotating diamond sigils reinforce the illusion/mischief look.
        for i in range(4):
            angle = elapsed * 0.7 + i * math.tau / 4
            cx = int(palm[0] + math.cos(angle) * 54)
            cy = int(palm[1] + math.sin(angle) * 40)
            r = 7
            diamond = [(cx, cy-r), (cx+r, cy), (cx, cy+r), (cx-r, cy)]
            poly(overlay, diamond, (75, 235, 100), 2, True)
        for i in range(5):
            angle = elapsed * 0.6 + i * math.tau / 5
            end = (int(palm[0] + math.cos(angle) * 66),
                   int(palm[1] + math.sin(angle) * 48))
            energy_wisp(overlay, palm, end, (30, 155, 45),
                        elapsed + i * 0.1, seed=360 + i, width=2)

    def _render_thor(self, overlay, palm, tips, elapsed, open_hand, fist):
        # Thor: a concentrated storm core with branching blue-white lightning.
        pulse = 1.0 + 0.18 * math.sin(elapsed * 10.0)
        glow(overlay, palm, int(19 * pulse), (255, 170, 35), (255, 255, 255))
        selected = tips if open_hand else [tips[1]]
        for i, tip in enumerate(selected):
            electric_bolt(overlay, palm, tip, (255, 205, 65),
                          int(elapsed * 28) + i * 37, 22, 3, 4)
        # A dominant upward strike and side forks make the effect read as thunder.
        targets = [
            (palm[0] - 18, max(8, palm[1] - 150)),
            (palm[0] + 8, max(8, palm[1] - 185)),
            (palm[0] + 38, max(8, palm[1] - 135)),
        ]
        for i, end in enumerate(targets):
            electric_bolt(overlay, palm, end, (255, 225, 125),
                          int(elapsed * 31) + 520 + i * 19, 27, 3, 4)
        # Short electric arcs orbit the wrist/palm between larger strikes.
        for i in range(3):
            angle = elapsed * 1.4 + i * math.tau / 3
            end = (int(palm[0] + math.cos(angle) * 57),
                   int(palm[1] + math.sin(angle) * 42))
            electric_bolt(overlay, palm, end, (255, 185, 65),
                          int(elapsed * 24) + 610 + i, 10, 1, 2)

    def _render_thanos(self, overlay, palm, tips, elapsed, open_hand, points, width, height):
        """Purple cosmic energy and an Infinity-Gauntlet-inspired hand overlay."""
        pulse = 1.0 + 0.12 * math.sin(elapsed * 4.5)
        glow(overlay, palm, int(22 * pulse), (155, 65, 205), (235, 205, 255))

        # Six Infinity-Stone coloured lights orbit the palm.
        stone_colors = [
            (30, 30, 235), (30, 190, 245), (210, 55, 190),
            (55, 210, 70), (220, 150, 35), (190, 80, 220)
        ]
        for i, color in enumerate(stone_colors):
            angle = elapsed * 0.45 + i * math.tau / 6
            x = int(palm[0] + math.cos(angle) * 43)
            y = int(palm[1] + math.sin(angle) * 34)
            glow(overlay, (x, y), 5, color, (255, 255, 255))

        # Purple energy filaments extend from the palm toward the fingers.
        selected = tips[1:] if open_hand else [tips[1]]
        for i, tip in enumerate(selected):
            energy_wisp(overlay, palm, tip, (165, 75, 225),
                        elapsed + i * 0.12, seed=820 + i, width=3)

        # A gold cuff at the wrist gives the hand a gauntlet-like silhouette.
        wrist = to_px(points[0, :2], width, height)
        cv2.circle(overlay, wrist, 15, (35, 175, 235), 4, cv2.LINE_AA)
        cv2.circle(overlay, wrist, 10, (80, 215, 255), 2, cv2.LINE_AA)
        glow(overlay, palm, 5, (175, 80, 225), (255, 245, 255))

    def render_hand(self, frame, character: Character, hand, elapsed, hand_id=0):
        height, width = frame.shape[:2]
        points = hand["points"]
        palm = to_px(palm_center(points), width, height)
        tips = [to_px(points[i, :2], width, height) for i in (4, 8, 12, 16, 20)]
        open_hand = hand.get("gesture") == "Open_Palm" or hand_is_open(points)
        fist = hand.get("gesture") == "Closed_Fist"
        pinch = is_pinching(points)
        power = 1.0 if open_hand else (0.82 if fist else 0.68)
        overlay = np.zeros_like(frame)

        name = character.name
        if name == "DOCTOR STRANGE":
            self._render_strange(overlay, palm, elapsed, power)
        elif name == "SCARLET WITCH":
            self._render_wanda(overlay, palm, tips, elapsed, open_hand)
        elif name == "AGATHA HARKNESS":
            self._render_agatha(overlay, palm, tips, elapsed, open_hand)
        elif name == "LOKI":
            self._render_loki(overlay, palm, tips, elapsed, open_hand)
        elif name == "THOR":
            self._render_thor(overlay, palm, tips, elapsed, open_hand, fist)
        elif name == "THANOS":
            self._render_thanos(overlay, palm, tips, elapsed, open_hand, points, width, height)

        # Pinch focuses the existing effect at the fingertips; it does not add
        # another generic ring around the hand.
        if pinch:
            focus = ((tips[0][0] + tips[1][0]) // 2,
                     (tips[0][1] + tips[1][1]) // 2)
            glow(overlay, focus, 5, character.accent, (255, 255, 255))

        bloom_composite(frame, overlay, blur=12, bloom_strength=0.72,
                        detail_strength=0.92)

    def render_two_hand_field(self, frame, character: Character, hands, elapsed):
        if len(hands) < 2:
            return

        height, width = frame.shape[:2]
        p1, p2 = hands[0]["points"], hands[1]["points"]
        c1 = to_px(palm_center(p1), width, height)
        c2 = to_px(palm_center(p2), width, height)
        midpoint = ((c1[0] + c2[0]) // 2, (c1[1] + c2[1]) // 2)
        distance = math.dist(c1, c2)
        overlay = np.zeros_like(frame)
        name = character.name

        if name == "AGATHA HARKNESS":
            # Main purple beam between both palms, with a luminous core and
            # two restrained flowing edge filaments.
            cv2.line(overlay, c1, c2, (220, 35, 245), 23, cv2.LINE_AA)
            cv2.line(overlay, c1, c2, (245, 120, 255), 10, cv2.LINE_AA)
            cv2.line(overlay, c1, c2, (255, 245, 255), 3, cv2.LINE_AA)
            energy_wisp(overlay, c1, c2, (230, 65, 255), elapsed, 701, 3)
            energy_wisp(overlay, c2, c1, (210, 45, 245), elapsed + 0.5, 702, 2)
            glow(overlay, c1, 13, (220, 35, 245))
            glow(overlay, c2, 13, (220, 35, 245))

        elif name == "SCARLET WITCH":
            # Red energy surges between the hands as a turbulent crimson stream.
            cv2.line(overlay, c1, c2, (0, 0, 190), 19, cv2.LINE_AA)
            energy_wisp(overlay, c1, c2, (0, 0, 245), elapsed, 711, 5)
            energy_wisp(overlay, c2, c1, (0, 0, 220), elapsed + 0.4, 712, 3)
            glow(overlay, c1, 16, (0, 0, 245))
            glow(overlay, c2, 16, (0, 0, 245))

        elif name == "THOR":
            # Lightning arcs leap through the space between the hands.
            for i in range(5):
                electric_bolt(overlay, c1, c2, (255, 215, 95),
                              int(elapsed * 24) + 800 + i * 17,
                              25 + i * 3, 2, 3 if i == 0 else 2)
            glow(overlay, c1, 13, (255, 195, 65))
            glow(overlay, c2, 13, (255, 195, 65))

        elif name == "LOKI":
            for i in range(3):
                electric_bolt(overlay, c1, c2, (45, 210, 65),
                              int(elapsed * 17) + 900 + i * 23,
                              19 + i * 4, 2, 2)
            glow(overlay, midpoint, 8, (45, 220, 65))

        elif name == "DOCTOR STRANGE":
            radius = int(np.clip(distance * 0.48, 55, min(width, height) * 0.34))
            mandala(overlay, midpoint, radius, (0, 155, 255),
                    (80, 220, 255), elapsed)

        elif name == "THANOS":
            for i, color in enumerate(((155, 65, 205), (80, 120, 235), (35, 190, 225))):
                energy_wisp(overlay, c1, c2, color, elapsed + i * 0.22, 850 + i, 3)
            glow(overlay, midpoint, 18, (155, 65, 205), (245, 220, 255))

        bloom_composite(frame, overlay, blur=16, bloom_strength=0.85,
                        detail_strength=0.90)

Overwriting src/vfx.py


In [6]:
%%writefile src/main.py
import argparse
import time
import random

import cv2
import numpy as np

from config import CHARACTERS
from hand_tracking import HandTracker, PersonSegmenter
from vfx import MagicRenderer


class SnapDetector:
    """Detect a quick thumb-index pinch followed by a sharp release."""

    def __init__(self):
        self.pinched_at = {}
        self.last_snap = 0.0
        self.previous_distance = {}

    def update(self, hands, now):
        snapped = False
        for hand_id, hand in enumerate(hands):
            points = hand["points"]
            distance = float(np.linalg.norm(points[4, :2] - points[8, :2]))
            was_pinched = hand_id in self.pinched_at
            if distance < 0.045:
                self.pinched_at[hand_id] = now
            elif was_pinched:
                held = now - self.pinched_at.pop(hand_id)
                previous = self.previous_distance.get(hand_id, distance)
                quick_release = distance > 0.105 and held < 0.55
                sharp_motion = (distance - previous) > 0.035
                if quick_release and sharp_motion and now - self.last_snap > 1.0:
                    snapped = True
                    self.last_snap = now
            self.previous_distance[hand_id] = distance

        # Forget stale pinches when a hand disappears from the camera.
        active_ids = set(range(len(hands)))
        for hand_id in list(self.pinched_at):
            if hand_id not in active_ids and now - self.pinched_at[hand_id] > 0.6:
                self.pinched_at.pop(hand_id, None)
        return snapped


def draw_hud(frame, character, hands, fps, show_help=True, vanished=False):
    h, w = frame.shape[:2]
    panel = frame.copy()
    cv2.rectangle(panel, (0, 0), (w, 88), (12, 10, 24), -1)
    cv2.addWeighted(panel, 0.74, frame, 0.26, 0, frame)
    cv2.putText(frame, "MYSTIC MOTION", (18, 31),
                cv2.FONT_HERSHEY_DUPLEX, 0.78, (235, 235, 255), 2, cv2.LINE_AA)
    cv2.putText(frame, character.name, (18, 62),
                cv2.FONT_HERSHEY_SIMPLEX, 0.58, character.accent, 2, cv2.LINE_AA)
    status = f"HANDS: {len(hands)}   FPS: {fps:.0f}"
    cv2.putText(frame, status, (w - 230, 36),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (235, 235, 245), 1, cv2.LINE_AA)
    if character.name == "THANOS":
        state = "SNAPPED / VANISHED" if vanished else "READY - PINCH & FLICK TO SNAP"
        cv2.putText(frame, state, (w - 440, 68),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                    (185, 100, 245) if vanished else (225, 220, 245), 1, cv2.LINE_AA)
    if show_help:
        help_text = "1 Strange  2 Wanda  3 Agatha  4 Loki  5 Thor  6 Thanos"
        cv2.putText(frame, help_text, (16, h - 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.46, (210, 215, 235), 1, cv2.LINE_AA)
        cv2.putText(frame, "Q: quit | H: help | S: hand skeleton",
                    (16, h - 42), cv2.FONT_HERSHEY_SIMPLEX, 0.42,
                    (180, 185, 205), 1, cv2.LINE_AA)


def draw_skeleton(frame, points):
    h, w = frame.shape[:2]
    p = [(int(x * w), int(y * h)) for x, y, _ in points]
    connections = [
        (0,1),(1,2),(2,3),(3,4), (0,5),(5,6),(6,7),(7,8),
        (5,9),(9,10),(10,11),(11,12), (9,13),(13,14),(14,15),(15,16),
        (13,17),(17,18),(18,19),(19,20),(0,17)
    ]
    for a, b in connections:
        cv2.line(frame, p[a], p[b], (110, 210, 180), 1, cv2.LINE_AA)
    for point in p:
        cv2.circle(frame, point, 2, (220, 245, 235), -1, cv2.LINE_AA)


def apply_snap_disappearance(frame, segmenter, progress, elapsed):
    """Blend the segmented person into an inpainted background and add dust."""
    mask_float = segmenter.mask(frame)
    if mask_float is None:
        return frame
    person_mask = (mask_float > 0.35).astype(np.uint8) * 255
    if cv2.countNonZero(person_mask) == 0:
        return frame

    # Inpainting approximates the background behind the segmented person.
    # It works best with a simple/consistent room background.
    clean_background = cv2.inpaint(frame, person_mask, 9, cv2.INPAINT_TELEA)
    alpha = (mask_float * progress).clip(0.0, 1.0)[..., None]
    result = (frame.astype(np.float32) * (1.0 - alpha) +
              clean_background.astype(np.float32) * alpha).astype(np.uint8)

    # A brief purple-gold particle burst follows the dissolving silhouette.
    if progress > 0.02:
        rng = random.Random(int(elapsed * 1000))
        ys, xs = np.where(person_mask > 0)
        if len(xs):
            overlay = np.zeros_like(frame)
            count = min(260, max(40, len(xs) // 110))
            indices = rng.sample(range(len(xs)), min(count, len(xs)))
            for idx in indices:
                x, y = int(xs[idx]), int(ys[idx])
                radius = rng.choice((1, 2, 3))
                color = rng.choice(((155, 75, 220), (210, 165, 65), (235, 220, 255)))
                cv2.circle(overlay, (x, y), radius, color, -1, cv2.LINE_AA)
            result = cv2.addWeighted(result, 1.0, overlay, 0.9, 0)
    return result


def run(camera_index=0, width=1280, height=720, mirror=True, show_skeleton=False):
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError(
            f"Could not open camera index {camera_index}. "
            "Check macOS camera permission or try camera_index=1."
        )

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)

    tracker = None
    segmenter = None
    try:
        tracker = HandTracker(max_hands=2)
        renderer = MagicRenderer()
        snap_detector = SnapDetector()

        character_key = "1"
        show_help = True
        previous = time.perf_counter()
        fps = 0.0
        vanished = False
        transition_start = None
        transition_from = 0.0
        transition_to = 0.0
        transition_duration = 1.25

        print("\nMYSTIC MOTION started.")
        print("Select characters using keys 1-6.")
        print("In Thanos mode: pinch thumb + index finger, then quickly flick apart.")
        print("H = Help | S = Skeleton | Q = Quit\n")

        while True:
            ok, frame = cap.read()
            if not ok:
                print("Camera frame could not be read. Stopping.")
                break
            if mirror:
                frame = cv2.flip(frame, 1)

            now = time.perf_counter()
            dt = max(now - previous, 1e-6)
            previous = now
            current_fps = 1.0 / dt
            fps = 0.9 * fps + 0.1 * current_fps if fps else current_fps
            elapsed = now

            hands = tracker.process(frame)
            character = CHARACTERS[character_key]

            if character.name == "THANOS":
                if segmenter is None:
                    segmenter = PersonSegmenter()
                if snap_detector.update(hands, now):
                    transition_from = 1.0 if vanished else 0.0
                    transition_to = 0.0 if vanished else 1.0
                    transition_start = now
                    vanished = not vanished

                if transition_start is not None:
                    fraction = min(1.0, (now - transition_start) / transition_duration)
                    eased = fraction * fraction * (3.0 - 2.0 * fraction)
                    progress = transition_from + (transition_to - transition_from) * eased
                    frame = apply_snap_disappearance(frame, segmenter, progress, elapsed)
                    if fraction >= 1.0:
                        transition_start = None
                elif vanished:
                    frame = apply_snap_disappearance(frame, segmenter, 1.0, elapsed)
            else:
                # Switching away from Thanos restores the live camera image.
                vanished = False
                transition_start = None

            for hand_id, hand in enumerate(hands):
                if show_skeleton:
                    draw_skeleton(frame, hand["points"])
                renderer.render_hand(frame, character, hand, elapsed, hand_id)

            if len(hands) >= 2:
                renderer.render_two_hand_field(frame, character, hands, elapsed)

            draw_hud(frame, character, hands, fps, show_help, vanished)
            cv2.imshow("MYSTIC MOTION | Live Magic Experience", frame)

            key = cv2.waitKey(1) & 0xFF
            if key in (ord("q"), ord("Q"), 27):
                print("Closing Mystic Motion...")
                break
            elif key in (ord("h"), ord("H")):
                show_help = not show_help
            elif key in (ord("s"), ord("S")):
                show_skeleton = not show_skeleton
            elif ord("1") <= key <= ord("6"):
                character_key = chr(key)
                print("Selected:", CHARACTERS[character_key].name)
                if CHARACTERS[character_key].name != "THANOS":
                    vanished = False
                    transition_start = None

    finally:
        if tracker is not None:
            tracker.close()
        if segmenter is not None:
            segmenter.close()
        cap.release()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Mystic Motion real-time hand VFX")
    parser.add_argument("--camera", type=int, default=0)
    parser.add_argument("--width", type=int, default=1280)
    parser.add_argument("--height", type=int, default=720)
    parser.add_argument("--no-mirror", action="store_true")
    parser.add_argument("--skeleton", action="store_true")
    args = parser.parse_args()
    run(
        camera_index=args.camera,
        width=args.width,
        height=args.height,
        mirror=not args.no_mirror,
        show_skeleton=args.skeleton,
    )


Overwriting src/main.py


In [7]:
import ast
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

for filename in ("config.py", "hand_tracking.py", "vfx.py", "main.py"):
    ast.parse((SRC / filename).read_text(encoding="utf-8"))
    print(f"Syntax OK: {filename}")

import config
import hand_tracking
import vfx
import main

for module in (config, hand_tracking, vfx, main):
    importlib.reload(module)

assert len(config.CHARACTERS) == 6, f"Expected 6 characters, got {len(config.CHARACTERS)}"
assert [c.name for c in config.CHARACTERS.values()] == [
    "DOCTOR STRANGE", "SCARLET WITCH", "AGATHA HARKNESS", "LOKI", "THOR", "THANOS"
]
assert hasattr(hand_tracking, "PersonSegmenter")
assert hasattr(main, "SnapDetector")
assert hasattr(vfx.MagicRenderer, "_render_thanos")
assert hasattr(vfx.MagicRenderer, "render_two_hand_field")

print("\nALL CHECKS PASSED")
print("Characters:", ", ".join(c.name for c in config.CHARACTERS.values()))


Syntax OK: config.py
Syntax OK: hand_tracking.py
Syntax OK: vfx.py
Syntax OK: main.py



ALL CHECKS PASSED
Characters: DOCTOR STRANGE, SCARLET WITCH, AGATHA HARKNESS, LOKI, THOR, THANOS


In [8]:
from main import run

run(
    camera_index=0,
    width=1280,
    height=720,
    mirror=True,
    show_skeleton=False,
)


I0000 00:00:1794496849.645679 1668166 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3
W0000 00:00:1794496849.646175 1668166 gesture_recognizer_graph.cc:129] Hand Gesture Recognizer contains CPU only ops. Sets HandGestureRecognizerGraph acceleration to Xnnpack.
I0000 00:00:1794496849.647230 1668166 hand_gesture_recognizer_graph.cc:250] Custom gesture classifier is not defined.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1794496849.657139 1701626 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794496849.664975 1701626 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794496849.665877 1701624 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback te


MYSTIC MOTION started.
Select characters using keys 1-6.
In Thanos mode: pinch thumb + index finger, then quickly flick apart.
H = Help | S = Skeleton | Q = Quit



/opt/anaconda3/envs/marvel-magic/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Selected: AGATHA HARKNESS
Selected: SCARLET WITCH
Closing Mystic Motion...
